# Seminar 7 — Part 1: Extracting J parameters from a machine-learned interatomic potential

In Seminar 6 you ran Monte Carlo simulations with an Ising-like energy model:

$$E = \sum_{n} J_n \sum_{\langle i,j \rangle_n} s_i s_j$$

where $J_n$ was a free parameter you chose by hand.  
Today we calculate $J_n$ from first-principles energetics using **Orb v2**, a machine-learned interatomic potential trained on DFT data.

The key idea is that the Ising energy is **linear in the $J_n$**, so we can fit them by
computing the total energy for several different atomic configurations and solving a small linear regression problem.

### Workflow
1. Set up your lattice (same `cell` and `sites` as Seminar 6).
2. Load Orb v2 and verify it works.
3. **(Optional / 3a)** Manually create a few structures in VESTA, calculate their energies, and do a by-hand fit — to understand the principle.
4. **(3b)** Automatically generate configurations, calculate energies with Orb v2, and fit $J_1, J_2$ by least squares.
5. Use the fitted $J$ values in your Monte Carlo simulation (Notebook 2).

## Setup

Run the cell below.  
- **Local installation**: if `orb-models`, `gemmi`, and `ase` are already installed, nothing extra happens.
- **Google Colab fallback**: if they are missing the cell installs them and clones the helper code.

In [2]:
import importlib, subprocess, sys, os

def _pkg_missing(name):
    return importlib.util.find_spec(name) is None

if any(_pkg_missing(p) for p in ['orb_models', 'gemmi', 'ase', 'h5py']):
    print('Installing packages (Colab / first run) ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'orb-models', 'ase', 'gemmi', 'h5py'], check=True)
    # Clone helper repository if not already present
    if not os.path.exists('seminar_disorder_2026'):
        subprocess.run(['git', 'clone', '-q',
                        'https://github.com/aglie/seminar_disorder_2026.git'], check=True)
    os.chdir('seminar_disorder_2026')
    print('Done.')
else:
    print('All libraries found locally.')

import numpy as np
import matplotlib.pyplot as plt

from mc_helpers import (
    calculate_connectivity_lists,
    initialise,
    calc_E,
    monte_carlo_run,
    convert_spins_to_crystal_structure,
    ising_features,
)

print('Imports OK.')

All libraries found locally.
Imports OK.


## Section 1: Set up your lattice

Use the **same `cell` and `sites`** parameters as you used in Seminar 6 for your chosen disordered structure.  
If you have not decided yet, the BCC example below is a good starting point.

We use a **small supercell** here (`box_small = [2,2,2]`) because we only need it for energy calculations with Orb v2.

> **Your structure**: identify the disordered site in your CIF from the homework.  The two species on that site become your spin-up (+1) and spin-down (-1) atoms.

### Static atoms

Many real structures contain atoms that are **not disordered** — they sit on a fixed site in every unit cell (e.g. the oxygen framework in a perovskite, the host cation in a doped material).  These atoms do not need to be in the MC simulation (they contribute only to Bragg scattering, not diffuse scattering), but they **do** need to be present when Orb v2 calculates the energy, because they create the local chemical environment that determines $J$.

Set `static_atoms` to a list of `(element, frac_x, frac_y, frac_z)` for each non-disordered site in one unit cell.  Leave it as `None` if every atom on every site is part of the disorder (e.g. the BCC example).

In [ ]:
# ── Edit these to match your structure ──────────────────────────────────────
# Unit cell parameters  [a(Å), b(Å), c(Å), alpha(°), beta(°), gamma(°)]
cell = np.array([4.0, 4.0, 4.0, 90., 90., 90.])

# Fractional coordinates of the DISORDERED site(s) within one unit cell
sites = np.array([[0.0, 0.0, 0.0],    # e.g. BCC corner
                  [0.5, 0.5, 0.5]])   #      BCC body centre

# Element symbols: spin +1 → up_element, spin -1 → down_element
up_element   = 'Au'
down_element = 'Cu'

# Non-disordered atoms in one unit cell: list of (element, frac_x, frac_y, frac_z)
# These are added to every Orb v2 calculation but NOT to the MC simulation.
# Set to None if all atoms on all sites are part of the disorder.
static_atoms = None
# Example — perovskite ABO3 with B-site disorder:
# static_atoms = [
#     ('Ba', 0.0, 0.0, 0.0),          # A-site cation
#     ('O',  0.5, 0.5, 0.0),          # three oxygens per unit cell
#     ('O',  0.5, 0.0, 0.5),
#     ('O',  0.0, 0.5, 0.5),
# ]

# Small supercell for energy calculations (keep this small — 2×2×2 is enough)
box_small = np.array([2, 2, 2], dtype=int)

# Number of interaction shells to fit (start with 2)
N_shells = 2
J_placeholder = np.zeros(N_shells)   # we will fit these below
# ────────────────────────────────────────────────────────────────────────────

connectivity_lists, frac_coords, dists, counts, N_ats = calculate_connectivity_lists(
    cell, sites, box_small, J_placeholder
)

n_static = len(static_atoms) * int(box_small.prod()) if static_atoms else 0
print(f'Supercell: {box_small[0]}×{box_small[1]}×{box_small[2]}')
print(f'  Disordered atoms: {N_ats}')
print(f'  Static atoms:     {n_static}')
print(f'  Total for Orb:    {N_ats + n_static}')
print()
print('Distance (Å)   avg. neighbours')
for d, n in zip(dists[:6], counts[:6]):
    print(f'  {d:8.4f}         {n:.1f}')

## Section 2: Load Orb v2

Orb v2 is a **machine-learned interatomic potential** trained on a large DFT dataset.  It predicts the total energy (and forces) of any atomic structure at near-DFT quality in milliseconds.

We use it here as a black box: give it an atomic structure, get back a total energy in eV.

In [ ]:
from orb_models.forcefield import pretrained
from orb_models.forcefield.calculator import ORBCalculator

device = 'cpu'   # change to 'cuda' if you have a GPU
# Note: the model file (~100 MB) is downloaded on first run — this takes ~1 minute.
orbff  = pretrained.orb_v2(device=device)
calc   = ORBCalculator(orbff, device=device)

print('Orb v2 loaded.')

# Quick smoke test: all-up configuration (with static atoms if defined)
spins_test, _ = initialise(N_ats, 1.0, J_placeholder, connectivity_lists)
crystal_test  = convert_spins_to_crystal_structure(
    frac_coords, spins_test, cell, box_small,
    up_element=up_element, down_element=down_element,
    static_atoms=static_atoms,
)
atoms_test = crystal_test.to_ase_atoms()
atoms_test.calc = calc
E_test = atoms_test.get_potential_energy()
print(f'Energy of all-{up_element} + static atoms: {E_test:.4f} eV  ({len(atoms_test)} atoms total)')

## The linear model for J

The Ising energy (in Seminar 6 units) for a spin configuration $\{s_i\}$ is:

$$E_\text{Ising} = \sum_n J_n \, f_n(\{s_i\})$$

where the **feature** $f_n$ is the quantity `calc_E` computes when $J_n=1$ for shell $n$ and zero for all others:

$$f_n = \frac{1}{N} \sum_i s_i \sum_{j \in \text{NN}_n(i)} s_j$$

Because $f_n$ depends only on the geometry (not on $J$), we can compute it for any configuration and then fit $J_n$ by matching the Orb v2 total energies:

$$E_\text{Orb}(\text{config}) \approx E_0 + \sum_n J_n \, f_n(\text{config})$$

Collecting many configurations gives a linear system $\mathbf{y} = X \boldsymbol{\beta}$, solved by least squares.

---
## Section 3a (Optional): Manual fit in VESTA

This section walks through the fitting principle by hand, using structures you create yourself in VESTA.  **Skip to Section 3b if you prefer to go straight to the automated version.**

### Steps
1. Run the cell below to write a reference CIF (all one atom type) to disk.
2. Open it in VESTA.
3. Using **Edit → Edit data → Structure parameters**, change 1, 2, or 3 atoms from `up_element` to `down_element`.  Save each as a new CIF (`config_1swap.cif`, `config_2swaps_near.cif`, etc.).
4. Load those CIFs in the cell that follows, compute their energies with Orb v2, and fill in the pair-count table.
5. Do a quick least-squares fit and read off $J_1$.

**Question**: when you have two swapped atoms, does it matter whether they are nearest neighbours or further apart?  What does the energy difference tell you?

In [ ]:
# Write the all-up reference CIF for VESTA editing
os.makedirs('vesta_configs', exist_ok=True)
spins_ref = np.ones(N_ats, dtype=int)
crystal_ref = convert_spins_to_crystal_structure(
    frac_coords, spins_ref, cell, box_small,
    up_element=up_element, down_element=down_element,
    static_atoms=static_atoms,
)
crystal_ref.save_cif('vesta_configs/reference_all_up.cif')
print('Saved: vesta_configs/reference_all_up.cif')
print(f'Open this in VESTA, swap atoms from {up_element} to {down_element}, save new CIFs.')

In [ ]:
# ── Load your VESTA-edited CIFs and compute their Orb energies ──────────────
# Edit the list below with the filenames you created.
from ase.io import read as ase_read

vesta_cifs = [
    'vesta_configs/reference_all_up.cif',
    'vesta_configs/config_1swap.cif',
    # 'vesta_configs/config_2swaps_near.cif',
    # 'vesta_configs/config_2swaps_far.cif',
]

vesta_energies = []
for path in vesta_cifs:
    atoms = ase_read(path)
    atoms.calc = calc
    E = atoms.get_potential_energy()
    vesta_energies.append(E)
    print(f'{path}: E = {E:.4f} eV')

In [ ]:
# ── Fill in the pair counts manually (count in VESTA using Show Bonds) ──────
# N1_unlike[k] = number of nearest-neighbour unlike pairs in config k
# Count only unique pairs: (A,B) counts once, not twice.

N1_unlike_manual = np.array([
    0,    # all-up: zero unlike pairs
    # ??,  # 1 swap
    # ??,  # 2 swaps, near
    # ??,  # 2 swaps, far
])

# Simple 1-shell fit: ΔE = c1 * N1_unlike  →  J1 = fitted slope
if len(vesta_energies) >= 2:
    vesta_E = np.array(vesta_energies)
    dE = vesta_E - vesta_E[0]     # energy relative to reference

    # Fit: dE = c1 * N1_unlike  (no intercept — reference has dE=0, N1=0)
    # The relationship between c1 and J1 is worked out in the markdown below.
    X_manual = N1_unlike_manual[1:].reshape(-1, 1)
    y_manual = dE[1:]
    c1_manual = np.linalg.lstsq(X_manual, y_manual, rcond=None)[0][0]
    print(f'Slope c1 = {c1_manual:.6f} eV / unlike pair')
    print('(Relationship to J1 is derived in Section 3b below.)')
else:
    print('Add more CIF files above to perform the fit.')

---
## Section 3b: Automated fit

We now generate configurations automatically using the MC code, compute their energies with Orb v2, and fit $J_1, J_2$ by least squares.

The feature vector for each configuration is computed by `ising_features()`, which calls `calc_E` with unit $J$ vectors — so the fitted coefficients are directly the $J_n$ values **in eV**, ready to paste into your MC simulation.

In [ ]:
np.random.seed(0)

# ── Why only 50/50 configurations? ───────────────────────────────────────────
# The Ising model treats both species symmetrically, so it cannot account for
# the difference in self-energy between e.g. all-Au and all-Cu.  Mixing those
# two reference points into the regression biases E₀ and gives a poor J fit.
# By fixing the composition at 50/50 the self-energy contribution is constant
# and absorbed into E₀, leaving only the ordering-dependent J terms to fit.
# (See also the note at the end of this section.)
# ─────────────────────────────────────────────────────────────────────────────

configs       = []
config_labels = []

# --- Extreme-ordering configurations (works for any lattice) -----------------
# Run a few MC steps at low T with a test J to push toward ordered / segregated,
# then use these configs as extremes for the fit.
J_test_pos = np.array([1.0] + [0.0] * (N_shells - 1))   # like-like preferred
J_test_neg = np.array([-1.0] + [0.0] * (N_shells - 1))  # unlike preferred

s_ord, _ = initialise(N_ats, 0.5, J_placeholder, connectivity_lists)
s_ord, _ = monte_carlo_run(s_ord, connectivity_lists, J_test_neg, T=0.1, n_moves=N_ats * 50)
configs.append(s_ord.copy())
config_labels.append('MC-ordered (unlike)')

s_seg, _ = initialise(N_ats, 0.5, J_placeholder, connectivity_lists)
s_seg, _ = monte_carlo_run(s_seg, connectivity_lists, J_test_pos, T=0.1, n_moves=N_ats * 50)
configs.append(s_seg.copy())
config_labels.append('MC-segregated (like)')

# --- Random 50/50 configurations (span the middle of the feature range) -----
N_random = 12
for k in range(N_random):
    s, _ = initialise(N_ats, 0.5, J_placeholder, connectivity_lists)
    configs.append(s.copy())
    config_labels.append(f'random-{k+1}')

print(f'Total configurations: {len(configs)}')
print(f'All have exactly {int(np.sum(configs[0] == 1))} spin-up and '
      f'{int(np.sum(configs[0] == -1))} spin-down atoms (50/50).')

In [ ]:
print('Calculating energies with Orb v2 ...')
orb_energies = []

for k, spins in enumerate(configs):
    crystal = convert_spins_to_crystal_structure(
        frac_coords, spins, cell, box_small,
        up_element=up_element, down_element=down_element,
        static_atoms=static_atoms,
    )
    atoms = crystal.to_ase_atoms()
    atoms.calc = calc
    E = atoms.get_potential_energy()
    orb_energies.append(E)
    print(f'  [{k+1:2d}/{len(configs)}]  {config_labels[k]:<20s}  E = {E:10.4f} eV')

orb_energies = np.array(orb_energies)

In [ ]:
# --- Compute Ising feature vectors for each configuration -------------------
features = np.array([
    ising_features(spins, N_ats, connectivity_lists)
    for spins in configs
])
# features[k, n] = f_n(config k)  — shape (N_configs, N_shells)

# --- Least-squares fit: E_orb = E0 + sum_n J_n * f_n ----------------------
X_fit = np.column_stack([np.ones(len(configs)), features])
y_fit = orb_energies

coeffs, residuals, rank, sv = np.linalg.lstsq(X_fit, y_fit, rcond=None)
E0_fit  = coeffs[0]
J_fit   = coeffs[1:]   # J1, J2, ...

y_pred = X_fit @ coeffs
ss_res = np.sum((y_fit - y_pred) ** 2)
ss_tot = np.sum((y_fit - y_fit.mean()) ** 2)
R2 = 1 - ss_res / ss_tot

print(f'E0 = {E0_fit:.4f} eV  (reference energy, not used in MC)')
for n, J in enumerate(J_fit):
    print(f'J{n+1} = {J:.6f} eV   (shell {n+1}, d = {dists[n+1]:.3f} Å, z = {counts[n+1]:.0f})')
print(f'\nFit quality  R² = {R2:.4f}')
print(f'RMSE         = {np.sqrt(ss_res/len(y_fit))*1000:.2f} meV')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Parity plot
ax = axes[0]
ax.scatter(y_pred, y_fit, alpha=0.7, edgecolors='k', linewidths=0.5)
lim = [min(y_fit.min(), y_pred.min()) - 0.05,
       max(y_fit.max(), y_pred.max()) + 0.05]
ax.plot(lim, lim, 'r--', lw=1)
ax.set_xlabel('Predicted energy  (eV)')
ax.set_ylabel('Orb v2 energy  (eV)')
ax.set_title(f'Ising fit  —  R² = {R2:.4f}')
ax.set_xlim(lim); ax.set_ylim(lim)

# Energy vs. first feature  (ΔE from mean)
ax = axes[1]
dE = y_fit - y_fit.mean()
ax.scatter(features[:, 0], dE, alpha=0.7, edgecolors='k', linewidths=0.5)
f1_range = np.linspace(features[:, 0].min(), features[:, 0].max(), 100)
ax.plot(f1_range, J_fit[0] * f1_range, 'r-', lw=1.5, label=f'slope = J₁ = {J_fit[0]:.4f} eV')
ax.set_xlabel('f₁  (NN Ising feature)')
ax.set_ylabel('ΔE from mean  (eV)')
ax.set_title('First-shell interaction')
ax.legend()

plt.tight_layout()
plt.show()

## Questions

1. **Sign of J₁**: is it positive or negative?  What does this predict about the ordering tendency of your system at low temperature?  Does this match chemical intuition (do the two atom types prefer to be together or segregated)?

2. **Magnitude of J₂ vs. J₁**: is the second-neighbour interaction significant?  At what point would you trust a NN-only model?

3. **Fit quality**: what is the R² of the Ising model fit?  If it is poor, what might be missing from the model?

4. **Connection to Section 3a** *(if you did it)*: the slope $c_1$ you measured in VESTA gave the energy per unlike bond.  Verify that $c_1$ is consistent with $J_1$ from the automated fit.  
   *Hint*: unlike bonds $N_1^\text{unlike}$ and the Ising feature $f_1$ are related by  
   $$\Delta E = -\frac{4 J_1}{N_\text{atoms}} \cdot N_1^\text{unlike}$$  
   so $c_1 = -4 J_1 / N_\text{atoms}$.

---
### Note: what if you want variable composition?

Here we fixed the composition at 50/50, which means the two end-member self-energies ($E(\text{all-A})$ and $E(\text{all-B})$) never enter the regression.  This is appropriate if you only want to describe ordering at a fixed composition.

If you wanted to model a system across **different compositions** (e.g. $x = 0.25, 0.5, 0.75$), you would need to account for the fact that the MLIP energy includes the self-energy of each species.  The standard approach is to compute the **energy above the convex hull**:

$$\Delta E_\text{hull}(x) = E(x) - \bigl[(1-x)\,E(\text{all-A}) + x\,E(\text{all-B})\bigr]$$

Fitting J to $\Delta E_\text{hull}$ instead of $E_\text{raw}$ removes the linear self-energy baseline, making the Ising model applicable across all compositions.  We skip this here because all students' structures show fixed-composition disorder (one atom replaced by another at the same site).

---
## Save J values for Notebook 2

In [ ]:
# Save the fitted J values to a file so Notebook 2 can load them
np.save('fitted_J.npy', J_fit)
print('Saved J values to fitted_J.npy')
print()
print('Summary for Notebook 2:')
print(f'  cell   = {list(cell)}')
print(f'  sites  = {sites.tolist()}')
print(f'  J_list = {J_fit.tolist()}')
print()
print('Copy the J_list line into the top of Notebook 2.')